# Sleep Quality on Productivity - Multiple Linear Regression (MLR)

### Imports and Loading Data

In [11]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from rulefit import RuleFit
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, RobustScaler, MaxAbsScaler, Normalizer, normalize
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error

sleepcycle_dataframe = pd.read_csv('..\\sleep_cycle_productivity.csv')

# Check that dataset is imported properly
sleepcycle_dataframe.head()

,Date,Person_ID,Age,Gender,Sleep Start Time,Sleep End Time,Total Sleep Hours,Sleep Quality,Exercise (mins/day),Caffeine Intake (mg),Screen Time Before Bed (mins),Work Hours (hrs/day),Productivity Score,Mood Score,Stress Level
0,2024-04-12,1860,32,Other,23.33,4.61,5.28,3,86,87,116,8.808920,8,3,6
1,2024-11-04,1769,41,Female,21.02,2.43,5.41,5,32,21,88,6.329833,10,3,7
2,2024-08-31,2528,20,Male,22.10,3.45,5.35,7,17,88,59,8.506306,10,9,10
3,2024-02-22,8041,37,Other,23.10,6.65,7.55,8,46,34,80,6.070240,8,4,2
4,2024-02-23,4843,46,Other,21.42,4.17,6.75,10,61,269,94,11.374994,8,7,9


In [15]:
X = sleepcycle_dataframe.drop(columns=['Productivity Score', 'Person_ID', 'Date'], axis=1)
y = sleepcycle_dataframe['Productivity Score']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the preprocessing steps for numerical and categorical features
numerical_columns = ['Total Sleep Hours', 'Sleep Start Time','Sleep Quality', 'Exercise (mins/day)', 'Caffeine Intake (mg)', 'Screen Time Before Bed (mins)', 
                     'Work Hours (hrs/day)', 'Stress Level']
categorical_columns = ['Gender']

In [13]:
RuleFit = RuleFit(tree_size=4, sample_fract=0.5, max_rules=1000, memory_par=True, random_state=42)

# Define the preprocessing steps for numerical and categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_columns),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_columns)
    ])

# Create a pipeline with preprocessing and the RuleFit model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('rulefit', RuleFit)
])

# Fit the pipeline to the training data
pipeline.fit(X_train, y_train)

# Make predictions on the test set
y_pred = pipeline.predict(X_test)

# Evaluate the model
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
print(f"RMSE: {rmse:.4f}")
print(f"R^2: {r2:.4f}")
print(f"MAE: {mae:.4f}")

c:\Users\Malik\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 5.544808261940489, tolerance: 2.172740622655662
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\Malik\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 5.586471954193257, tolerance: 2.172740622655662
  model = cd_fast.enet_coordinate_descent_gram(
c:\Users\Malik\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 22.371452489078365, tolerance: 2.172740622655662
  model = cd_fast.enet_coordinate_descent_gram(
c:\

RMSE: 2.8713
R^2: -0.0026
MAE: 2.5033


In [ ]:
# Plotting the feature importances
importances = pipeline.named_steps['rulefit'].feature_importances_
feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()
importances_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importances_df = importances_df[importances_df['Importance'] > 0].sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importances_df.head(10))
plt.title('Top 10 Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

# Plotting the predicted vs actual values
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title('Predicted vs Actual Sleep Quality')
plt.xlabel('Actual Sleep Quality')


TypeError: 'RuleFit' object is not callable